# Day 2 · 4교시 · 양자화 모델 평가

> **VLM 경량화 2일 과정 · Day 2 (4교시) · 실습**
> 실습 환경: **RunPod A100 80GB** · 커널: **`Python (vlm_quant)`** · 데이터: **HuggingFaceM4/ChartQA (test)**

---

## 이 교시의 목표
- test셋에서 **Relaxed Accuracy 4종**(원본/AWQ × 4B/8B)을 측정해 **양자화 손실**을 정량화한다.
- **모델 크기 · VRAM · 정확도**를 한 표로 비교한다.
- (선택) 공개 **8bit AWQ** 체크포인트를 4bit와 대조한다.
- 손실이 큰 케이스(세밀 차트·숫자)를 **분석**한다.


## 0. 공통 헤더 — RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN 로드

> 📌 **모든 Day 2 노트북은 이 셀을 먼저 실행합니다.** RunPod 영구 볼륨의 작업 폴더 **`/workspace/VLM_FT_2`** 를 기준으로 잡고, `.env`의 **HF_TOKEN**을 불러옵니다. (Day2는 RunPod이라 Google Drive 마운트가 없습니다.)
> - `VLM_DIR` / `DATA_DIR` : 교시 간 공유 폴더(모델·AWQ 결과·평가/벤치 JSON이 여기 모입니다).
> - **HF_TOKEN**: `VLM_FT_2/.env` 에 `HF_TOKEN=hf_...` 한 줄을 넣어두면 자동 로드됩니다. `login()`은 호출하지 않습니다(환경변수만으로 충분).

In [1]:
# ════════════════════════════════════════════════════════════
#  공통 헤더 · RunPod 작업 폴더(VLM_FT_2) + HF_TOKEN(.env) 로드
#  (모든 Day2 노트북 상단에서 동일하게 실행)
# ════════════════════════════════════════════════════════════
import os

# (1) RunPod 영구 볼륨의 작업 폴더 VLM_FT_2 (교시 간 모델·결과 공유). Drive 마운트 불필요.
VLM_DIR = '/workspace/VLM_FT_2'
DATA_DIR = f'{VLM_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# (2) .env 에서 HF_TOKEN 로드. login()은 부르지 않음(환경변수만으로 인증, 경고 없음).
try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv
ENV_PATH = f'{VLM_DIR}/.env'
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print('HF_TOKEN:', '로드됨' if os.environ.get('HF_TOKEN') else '없음')
else:
    print(f'.env 없음 — {ENV_PATH} 에 HF_TOKEN=hf_... 한 줄을 만들면 자동 로드됩니다(공개 모델만 쓰면 없어도 됨)')
print('작업 폴더 VLM_DIR =', VLM_DIR)

HF_TOKEN: 로드됨
작업 폴더 VLM_DIR = /workspace/VLM_FT_2


## 1. 평가 설계 — 4종 매트릭스

| | 4B | 8B |
|---|---|---|
| **원본(bf16)** | Day1 병합 4B | 베이스 8B |
| **AWQ(4bit)** | Day2-3 4B-AWQ | Day2-3 8B-AWQ |

같은 test 서브셋·같은 지표(Relaxed Accuracy)로 네 모델을 한 번에 비교합니다. 모델은 **하나씩 로드→평가→해제**(메모리 관리)합니다.

In [ ]:
# ── 공통 함수 + test셋 준비 ───────────────────────────────────
# 이 셀에서 수행할 작업
# 1) 필수 모듈 로드
# 2) HF_TOKEN 확인 및 (없으면) .env에 최초 저장
# 3) 평가 대상 모델 경로 정의
# 4) ChartQA test 서브셋 로드
# 5) Relaxed Accuracy 계산 함수 정의

import os, gc, torch
from datasets import load_dataset
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

# HF_TOKEN은 보통 상단 공통 헤더에서 .env 로드로 준비됨.
# 아직 환경변수에 없으면 최초 1회 토큰 입력을 받아 .env 파일에 저장.
if not os.environ.get('HF_TOKEN'):
    from getpass import getpass
    from dotenv import load_dotenv

    # 사용자 입력(화면에 값 숨김)
    tok = getpass('HF 토큰 입력(최초 1회): ').strip()

    # 공통 헤더에서 정의한 ENV_PATH(.env)에 토큰 저장
    with open(ENV_PATH, 'w') as f:
        f.write(f'HF_TOKEN={tok}\n')

    # 저장 직후 현재 세션 환경변수로 다시 로드
    load_dotenv(ENV_PATH)
    print('.env 생성:', ENV_PATH)

# 토큰 미존재 시 이후 HF 접근이 실패하므로 여기서 명시적으로 중단
assert os.environ.get('HF_TOKEN'), '.env 에 HF_TOKEN 이 없습니다'
print('HF 토큰 준비 완료 (.env 환경변수)')

# 작업 루트 디렉터리(공통 헤더에서 생성한 VLM_DIR 재사용)
WORKDIR = VLM_DIR

# 비교할 4개 모델 경로/식별자
MERGED_4B_DIR = f'{WORKDIR}/Qwen3-VL-4B-ChartQA-merged'  # Day2-3 병합본(원본 계열)
BASE_8B       = 'Qwen/Qwen3-VL-8B-Instruct'              # HF 허브의 원본 8B
AWQ_4B_DIR    = f'{WORKDIR}/Qwen3-VL-4B-ChartQA-AWQ'     # Day2-3 생성 4B AWQ
AWQ_8B_DIR    = f'{WORKDIR}/Qwen3-VL-8B-AWQ'             # Day2-3 생성 8B AWQ 경로

# 평가 샘플 수(시간/비용에 따라 조절 가능)
EVAL_N = 200

# ChartQA test split 로드 후:
# - seed 고정 셔플(재현성 확보)
# - 앞에서 EVAL_N개만 선택(평가 시간 단축)
test_set = (
    load_dataset('HuggingFaceM4/ChartQA', split='test')
    .shuffle(seed=42)
    .select(range(EVAL_N))
)

# 문자열을 숫자로 변환하는 헬퍼
# - 천 단위 콤마 제거
# - 퍼센트 기호 제거
# - 변환 실패 시 None 반환
def _to_number(s):
    s = s.strip().replace(',', '').replace('%', '').strip()
    try:
        return float(s)
    except ValueError:
        return None

def relaxed_match(pred, gold, tol=0.05):
    """
    Relaxed Accuracy 판정 함수
    - pred/gold가 모두 숫자로 해석되면:
      상대오차 <= tol(기본 5%)일 때 정답
    - 숫자가 아니면:
      소문자 기준 exact match
    """
    pred, gold = pred.strip(), gold.strip()
    pn, gn = _to_number(pred), _to_number(gold)

    if pn is not None and gn is not None:
        # 정답이 0인 경우 0으로 정확히 맞았는지 확인(0 나눗셈 방지)
        return pn == 0 if gn == 0 else abs(pn - gn) / abs(gn) <= tol

    # 텍스트 비교는 대소문자 무시
    return pred.lower() == gold.lower()

print('test 평가 샘플:', len(test_set))

HF 토큰 준비 완료 (.env 환경변수)
test 평가 샘플: 200


In [ ]:
# ── 모델 1개를 로드→평가→해제하고 (정확도, VRAM, 예측목록) 반환 ──

@torch.no_grad()  # 추론 시 그래디언트 비활성화(속도/메모리 절약)
def _predict(model, processor, image, question, max_new_tokens=32):
    """
    단일 샘플(이미지 + 질문)에 대한 모델 답변 생성 함수.
    반환값: 생성된 답변 문자열(str, 양끝 공백 제거)
    """
    # Qwen-VL 입력 포맷: user 메시지 안에 image + text를 함께 넣음
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': question}
        ]
    }]

    # chat template 적용 + 토크나이즈 + 텐서 변환 후 모델 디바이스로 이동
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,  # 답변 생성 시작 프롬프트 추가
        return_dict=True,
        return_tensors='pt'
    ).to(model.device)

    # 결정론적 생성(do_sample=False): 평가 재현성 확보
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    # 입력 토큰 이후(새로 생성된 부분)만 디코딩
    gen_ids = out[0][inputs['input_ids'].shape[1]:]
    return processor.decode(gen_ids, skip_special_tokens=True).strip()


def eval_model(path, dataset):
    """
    지정된 모델(path)을 로드해 dataset 전체를 평가.

    반환:
      (acc, vram, preds)
      - acc  : Relaxed Accuracy (float)
      - vram : 모델 로드 직후 GPU 메모리 사용량(GB, float)
      - preds: 샘플별 예측 문자열 리스트(list[str])
    """
    # GPU 피크 메모리 통계 초기화(필요 시 이후 별도 측정 가능)
    torch.cuda.reset_peak_memory_stats()

    # 모델/프로세서 로드
    # - dtype='auto'   : 체크포인트 타입(bf16/양자화 등) 자동 인식
    # - device_map='cuda': GPU에 배치
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        path,
        dtype='auto',
        device_map='cuda'
    ).eval()
    processor = AutoProcessor.from_pretrained(path)

    # 로드 직후 할당된 VRAM(GB) 기록
    vram = torch.cuda.memory_allocated() / 1024**3

    preds = []
    correct = 0

    # 데이터셋 순회 평가
    for ex in dataset:
        # 예측
        pred = _predict(model, processor, ex['image'], ex['query'])
        preds.append(pred)

        # 정답 비교(Relaxed Accuracy 기준)
        gold = str(ex['label'][0])
        if relaxed_match(pred, gold):
            correct += 1

    # 최종 정확도
    acc = correct / len(dataset)

    # 메모리 정리(다음 모델 평가를 위해 필수)
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

    return acc, vram, preds

## 2. 4종 모델 평가

하나씩 순서대로 평가합니다(각 모델 로드·해제로 시간이 걸립니다).

In [4]:
# ── 4종 평가 실행 ───────────────────────────────────────────
results = {}                       # 이름 → (acc, vram, preds)

print('· 4B 원본 평가...');  results['4B 원본'] = eval_model(MERGED_4B_DIR, test_set)
print('· 4B AWQ 평가...');   results['4B AWQ']  = eval_model(AWQ_4B_DIR, test_set)
print('· 8B 원본 평가...');  results['8B 원본'] = eval_model(BASE_8B, test_set)
print('· 8B AWQ 평가...');   results['8B AWQ']  = eval_model(AWQ_8B_DIR, test_set)
print('완료')

· 4B 원본 평가...


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

· 4B AWQ 평가...


Compressing model: 100%|██████████| 252/252 [00:00<00:00, 633.64it/s]


Loading weights:   0%|          | 0/1218 [00:00<?, ?it/s]

...

· 8B AWQ 평가...


Compressing model: 100%|██████████| 252/252 [00:00<00:00, 643.41it/s]


Loading weights:   0%|          | 0/1254 [00:00<?, ?it/s]

Decompressing model: 100%|██████████| 252/252 [00:00<00:00, 425.34it/s]


완료


평가 결과 

- Compressing model : config.json   "format": "pack-quantized", 로 패키징
- Decompressing model : 연산할 때 언패키징
-  252 ?    36 hiddenlayer * 7 (qkvo + MLP의 up down q) 

## 3. 종합 비교표 — 정확도 · 크기 · VRAM

In [ ]:
# ── 종합표 출력: 정확도 / 모델 크기 / VRAM 비교 ─────────────────────────────

def dir_size_gb(path):
    """디렉터리 전체 용량(GB) 계산."""
    # 경로가 없거나 디렉터리가 아니면 NaN 반환
    if not os.path.isdir(path):
        return float('nan')

    # 모든 하위 폴더의 파일 크기를 합산해 GB로 변환
    return sum(
        os.path.getsize(os.path.join(r, f))
        for r, _, fs in os.walk(path)
        for f in fs
        if os.path.isfile(os.path.join(r, f))
    ) / 1024**3


# 모델 크기(GB) 계산
# - 원본 모델: 파라미터 수 × 2 byte(bf16)로 근사
# - AWQ 모델: 실제 저장 디렉터리 크기 측정
sizes = {
    '4B 원본': 4_437_800_000 * 2 / 1024**3,
    '4B AWQ':  dir_size_gb(AWQ_4B_DIR),
    '8B 원본': 8_767_100_000 * 2 / 1024**3,
    '8B AWQ':  dir_size_gb(AWQ_8B_DIR),
}

# 결과 표 헤더 출력
print(f'{"모델":<10}{"Relaxed Acc":>12}{"크기(GB)":>12}{"VRAM(GB)":>11}')
print('-' * 46)

# 각 모델의 정확도/크기/VRAM 출력
for name in ['4B 원본', '4B AWQ', '8B 원본', '8B AWQ']:
    acc, vram, _ = results[name]  # results: (acc, vram, preds)
    print(f'{name:<10}{acc:>12.3f}{sizes[name]:>11.1f} {vram:>11.1f}')

print('-' * 46)

# 양자화 손실(원본 - AWQ) 출력
print(f"4B 양자화 손실: {results['4B 원본'][0] - results['4B AWQ'][0]:+.3f}")
print(f"8B 양자화 손실: {results['8B 원본'][0] - results['8B AWQ'][0]:+.3f}")


# ── 결과를 VLM_FT_2에 JSON으로 저장 (Day2-6 재사용용) ─────────────────────
import os as _os, json as _json

# 저장 디렉터리 보장
_os.makedirs(VLM_DIR, exist_ok=True)

# 저장할 핵심 지표만 추출
_summary = {
    name: {
        'relaxed_acc': round(results[name][0], 4),  # Relaxed Accuracy
        'load_vram_gb': round(results[name][1], 2), # 모델 로드 직후 VRAM
        'size_gb': round(sizes[name], 2),           # 모델 크기(GB)
    }
    for name in ['4B 원본', '4B AWQ', '8B 원본', '8B AWQ']
}

# 저장 파일 경로
_path = f'{VLM_DIR}/eval_results.json'

# JSON 저장 (한글 유지, 보기 좋게 들여쓰기)
with open(_path, 'w', encoding='utf-8') as _f:
    _json.dump(_summary, _f, ensure_ascii=False, indent=2)

print('평가 결과 저장:', _path)

모델         Relaxed Acc      크기(GB)   VRAM(GB)
----------------------------------------------
4B 원본            0.745        8.3         8.3
4B AWQ           0.710        4.0         3.3
8B 원본            0.000       16.3        16.3
8B AWQ           0.000        6.7         6.7
----------------------------------------------
4B 양자화 손실: +0.035
8B 양자화 손실: +0.000
평가 결과 저장: /workspace/VLM_FT_2/eval_results.json


## 4. 손실 분석 — AWQ가 틀린 케이스 들여다보기

**원본은 맞았는데 AWQ는 틀린** 샘플을 찾아, 양자화가 어떤 질문에서 무너지는지 봅니다(보통 세밀한 숫자 읽기).

In [7]:
# ── 8B: 원본 정답 & AWQ 오답인 샘플 추출 ────────────────────
# results에서 8B 원본/AWQ의 예측 목록만 꺼냄 (acc, vram은 _ 로 무시)
_, _, preds_orig = results['8B 원본']
_, _, preds_awq  = results['8B AWQ']

shown = 0
for i, ex in enumerate(test_set):
    gold = str(ex['label'][0])  # 정답 레이블을 문자열로 변환

    # 원본/AWQ 각각 정답 여부 판단 (Relaxed Accuracy 기준)
    ok_orig = relaxed_match(preds_orig[i], gold)
    ok_awq  = relaxed_match(preds_awq[i],  gold)

    # 원본은 맞고 AWQ는 틀린 경우만 출력 (양자화로 인해 깨진 케이스)
    if ok_orig and not ok_awq:
        print(f"Q: {ex['query']}")
        print(f"  정답   : {gold!r}")
        print(f"  원본   : {preds_orig[i]!r}  ✅")
        print(f"  8B AWQ : {preds_awq[i]!r}  ❌")
        print('-' * 60)
        shown += 1
        if shown >= 5: break  # 최대 5개만 출력

# 깨진 케이스가 하나도 없으면 양자화 손실이 거의 없다는 의미
if shown == 0:
    print('양자화로 깨진 케이스가 (이 서브셋에선) 없습니다 — 손실이 작다는 뜻.')

양자화로 깨진 케이스가 (이 서브셋에선) 없습니다 — 손실이 작다는 뜻.


## 5. 정리 + 다음 교시 예고

- **4종 Relaxed Accuracy**로 양자화 손실을 수치화했습니다. AWQ는 크기를 크게 줄이면서 정확도 손실은 **작게** 유지하는 경향을 확인했습니다.
- 크기·VRAM·정확도를 한 표로 비교하고, **AWQ가 깨지는 케이스**(세밀 숫자)를 들여다봤습니다.

### 다음 교시 — Day2-5 · vLLM 서빙 · 벤치
AWQ(compressed-tensors) 모델을 **vLLM**(별도 venv)으로 서빙하고, **OpenAI 호환 API**로 추론하며, 원본 대비 **처리량/지연/VRAM**을 측정합니다.

> ✅ **체크포인트**: ① 4종 정확도를 비교했다 ② 크기·VRAM 이득을 확인했다 ③ AWQ 손실 케이스를 봤다.